# How to Load Musical Themes Aligned to Recordings (MTD)

The Musical Theme Dataset (MTD; Zalkow, Balke, Arifi-Müller and Müller,
TISMIR 2020, [doi:10.5334/tismir.68](https://doi.org/10.5334/tismir.68))
takes 2,067 themes, each with its MTD ID, from Barlow and Morgenstern's *A
Dictionary of Musical Themes* and gives each one an encoding, an excerpt of a commercial
recording of its movement, and an alignment of the two: per note onset of
the encoding, the second of the excerpt where it sounds. This guide loads
one movement with `MtdLoader` and walks through what it makes:

1. per CD track, a seconds timeline, its length unknown unless you give it,
   with what MTD says of the recording as statements;
2. per theme, the excerpt, a child of the track at its start time;
3. per theme, the theme's encoding, a quarters timeline of its notes, in
   exact fractions of a quarter;
4. per aligned onset, a {{< glossary MatchClaim >}} from each note there to
   `CHUNK` on the excerpt: the excerpt holds no event, the claim states a
   position on it.

Finally the {{< glossary AlignmentBundle >}} is written as a package, a
TimeToShare manifest with its MatchClaim table beside it.

## Setup

Install the library with `pip install timetoalign`. The example data is a
sample of the MTD archive in its own layout, `MTD_sample.zip`, fetched by
the test-data helper; the full `MTD_1.0.0.zip` is read the same way, in
place, without extracting it. The helper `notes` shows a timeline's notes
one per row, and `claims` shows a bundle's MatchClaims from one timeline,
one per row: the note's onset and pitch, and the position the claim states
on the excerpt.

In [1]:
import tempfile
from fractions import Fraction
from pathlib import Path

import pandas as pd
import pyarrow.compute as pc

from timetoalign import AlignmentBundle, MtdLoader
from timetoalign.testdata import ensure_data

ARCHIVE = ensure_data("mtd") / "MTD_sample.zip"


def exact(column):
    """The exact values of a coordinate column."""
    numerators = pc.struct_field(column, "numerator").to_pylist()
    denominators = pc.struct_field(column, "denominator").to_pylist()
    return [Fraction(n, d) for n, d in zip(numerators, denominators)]


def notes(timeline):
    """A timeline's notes: onset and end in quarters, MIDI pitch."""
    table = timeline.get_events().table
    return pd.DataFrame(
        {
            "start": [str(value) for value in exact(table.column("start"))],
            "end": [str(value) for value in exact(table.column("end"))],
            "MIDI pitch": pc.struct_field(
                table.column("pitch"), "midi_number"
            ).to_pylist(),
        }
    )


def claims(bundle, timeline):
    """The bundle's MatchClaims from *timeline*: onset, pitch, the excerpt's second."""
    events = timeline.get_events().table
    pitch = dict(
        zip(
            events.column("id").to_pylist(),
            pc.struct_field(events.column("pitch"), "midi_number").to_pylist(),
        )
    )
    rows = []
    for claim in bundle.get_match_claims(timeline_id=timeline.id):
        rows.append(
            {
                "onset (quarters)": str(
                    Fraction(claim.start_anchor.coordinate_a.value)
                ),
                "MIDI pitch": pitch[claim.event_a_id],
                "excerpt side": "CHUNK" if claim.chunk_sides else claim.event_b_id,
                "excerpt (seconds)": claim.start_anchor.coordinate_b.value,
            }
        )
    return pd.DataFrame(rows)

## 1. Load one movement

A movement is named as MTD names it, `<ComposerID>_<WorkID>`; the first
movement of Beethoven's Piano Sonata Op. 2 No. 1 is
`Beethoven_Op002No1-01`. The loader reads every theme's metadata to find
the movement's themes, then only their alignments and encodings. Themes
can also be named by their MTD IDs, `themes=["MTD0807"]`; without either,
every theme is read.

In [2]:
loader = MtdLoader.from_file(ARCHIVE, movements=["Beethoven_Op002No1-01"])
loader

Sources,1 file(s): MTD_sample.zip
Claims,26
Movements,Beethoven_Op002No1-01
Themes,2
CD tracks,1
Theme encodings,data_EDM-corr_MID
Try,"create_timeline(), create_timelines(), create_bundle()"


Two themes of the movement, both in one CD track of Alfred Brendel's
recording. A track is named as MTD names it: the MusicBrainz release, the
disc and the track.

In [3]:
loader.themes, loader.tracks

(['MTD0807', 'MTD0808'],
 [('814c53fd-0732-4819-9753-4578f5ab992c', '049', 'Track01')])

The alignment counts the quarter notes of MTD's corrected encoding of the
theme from the Electronic Dictionary of Musical Themes (folder
`data_EDM-corr_MID`), so that is the encoding the loader reads by default.
The encoding of the engraved theme (`data_SCORE_MID`) can be chosen with
`theme_midi="data_SCORE_MID"`, but its notes differ from the counted ones
for 616 of the 2,067 themes (about 30 %): a row of the alignment meets no
onset, or the last row is not the end of the last note. The loader refuses
such a theme, and one refused theme refuses the whole load.

In [4]:
loader.theme_midi

'data_EDM-corr_MID'

## 2. The recording and its excerpts

`create_bundle()` assembles the bundle: the track's timeline, holding the
excerpts as children, and each theme's encoding as a timeline of its own.

In [5]:
bundle = loader.create_bundle()
bundle

AlignmentBundle(id='bundle_02o7mgugkagzf', name='Beethoven Op002No1-01', timelines=3, groups=0)

The bundle lists the tracks' timelines first, then the themes'. The track
is named after its movement, its performer, the disc and the track. MTD
does not state how long the track is, so its length is unknown, `None`.

In [6]:
track_id, *theme_ids = bundle.timeline_ids
track = bundle.get_timeline(track_id)
track.name, track.length

('Beethoven Op002No1-01, Brendel (CD 49, track 1)', None)

Each excerpt is a child of the track, placed at the theme's `StartTime`
and as long as `EndTime − StartTime`, both read as the decimals MTD
writes.

In [7]:
excerpts = [track.get_child(child_id) for child_id in track.list_children()]
pd.DataFrame(
    {
        "excerpt": [excerpt.name for excerpt in excerpts],
        "offset (seconds)": [track.get_child_offset(e.id).value for e in excerpts],
        "length (seconds)": [excerpt.length.value for excerpt in excerpts],
    }
)

,excerpt,offset (seconds),length (seconds)
0,Theme B689 (MTD 0807),0.2,2.8
1,Theme B690 (MTD 0808),84.7,4.5


The track states what MTD says of the recording: that it is a recording of
the movement, its release on MusicBrainz, the collection, disc and track as
MTD names them, the performer, and that the recording is in copyright and
identified here, not shared. No recording is read and no media file is
named. Each statement records how it was made: imported from MTD.

In [8]:
track.statements

{'dcterms:accessRights': [{'id': 'http://purl.org/coar/access_right/c_14cb'}],
 'dcterms:rights': [{'id': 'http://rightsstatements.org/vocab/InC/1.0/'}],
 'melod:hasIdentifier': [{'type': ['melod:Identifier'],
   'rdfs:label': ['MTD CollectionID'],
   'owl:hasValue': ['Composer/Beethoven_CompleteEdition_BC']},
  {'type': ['melod:Identifier'],
   'rdfs:label': ['MTD NameCD'],
   'owl:hasValue': ['049']},
  {'type': ['melod:Identifier'],
   'rdfs:label': ['MTD NameTrack'],
   'owl:hasValue': ['Track01']}],
 'rel:prf': [{'label': 'Brendel'}],
 'schema:inAlbum': [{'id': 'http://musicbrainz.org/release/814c53fd-0732-4819-9753-4578f5ab992c'}],
 'schema:recordingOf': [{'label': 'Beethoven Op002No1-01'}],
 'type': ['schema:MusicRecording']}

In [9]:
track.provenance[0]

{'property': 'rdf:type',
 'value': {'id': 'schema:MusicRecording'},
 'method': 'imported',
 'basis': ['https://doi.org/10.5334/tismir.68'],
 'note': 'MTD 1.0.0 data_META of MTD0807, MTD0808'}

A track length known from elsewhere, MusicBrainz for instance, is given to
`create_bundle` by the track's key (431 seconds here, for illustration):

In [10]:
(key,) = loader.tracks
loader.create_bundle(track_lengths={key: 431.0}).get_timeline(track.id).length

Coordinate(431.0, seconds)

## 3. The theme's encoding

The first theme's encoding is a quarters timeline of its own: never a
child of the excerpt, since the two are different manifestations of the
theme, related by the alignment. Each tick of the MIDI file (480 per
quarter) is an exact fraction: the triplet near the theme's end starts at
`20/3` quarters. The pitches are the encoding's: META's
`MidiTransposition`, the interval by which a recording sounds transposed
against the encoding (774 of the 2,067 themes, most an octave lower), is
not carried.

In [11]:
theme = bundle.get_timeline(theme_ids[0])
theme.name

'B689 (MTD 0807)'

In [12]:
notes(theme)

,start,end,MIDI pitch
0,0,479/480,60
1,1,959/480,65
2,2,1439/480,68
3,3,1919/480,72
4,4,2399/480,77
5,5,3119/480,80
6,13/2,3199/480,79
7,20/3,1093/160,77
8,41/6,3359/480,76
9,7,3839/480,77


Its statements name the movement it is part of, its MTD and Barlow and
Morgenstern numbers, and which encoding it is. A theme is not an incipit:
it is a part of the movement, wherever it stands.

In [13]:
theme.statements

{'dcterms:source': [{'label': "MTD's corrected EDM encoding of the theme, MTD 1.0.0 data_EDM-corr_MID/MTD0807_Beethoven_Op002No1-01.mid"}],
 'melod:hasIdentifier': [{'type': ['melod:Identifier'],
   'rdfs:label': ['MTD'],
   'owl:hasValue': ['0807']},
  {'type': ['melod:Identifier'],
   'rdfs:label': ['Barlow and Morgenstern'],
   'owl:hasValue': ['B689']}],
 'schema:isPartOf': [{'label': 'Beethoven Op002No1-01'}]}

## 4. The alignment as MatchClaims

Each row of the alignment file but the last is one MatchClaim: from each
note at that onset to `CHUNK` on the excerpt, anchored at the onset in
quarters and the excerpt's second. The file writes the onset with five
decimals (`6.66667`); the claim holds the exact onset of the note it
meets, `20/3`. The last row, the end of the last note, is checked and not
kept. The bundle holds the claims in its MatchClaim table, and
`get_match_claims` reads them back, here those of the first theme.

In [14]:
claims(bundle, theme)

,onset (quarters),MIDI pitch,excerpt side,excerpt (seconds)
0,0,60,CHUNK,0.12855
1,1,65,CHUNK,0.45519
2,2,68,CHUNK,0.75268
3,3,72,CHUNK,1.06608
4,4,77,CHUNK,1.38863
5,5,80,CHUNK,1.71300
6,13/2,79,CHUNK,2.13951
7,20/3,77,CHUNK,2.20401
8,41/6,76,CHUNK,2.28850
9,7,77,CHUNK,2.35321


From the triplet's onset, the bundle answers the excerpt's second and,
through the excerpt's offset, the track's. In the full dataset, 25 claims
state a second outside their excerpt (before its start or past its end);
they are kept as MTD gives them, and a matchstamp there names the
excerpt's second but none of the track, which the excerpt reaches only
within its span.

In [15]:
stamp = bundle.get_matchstamp_at(Fraction(20, 3), theme.id)
stamp

ID,Coordinate,Type
clt_0jong6cidqmuu,20/3 quarters,stated
cpt_2l49ja5agoizs,2.40401 seconds,inferred
cpt_3088r3rqtgx4l,2.20401 seconds,stated


## 5. Write the package

`to_file()` writes the manifest and, beside it, the MatchClaim table
(`match_claims.parquet`), which the manifest names among its resources;
`from_file()` reads both back.

In [16]:
package = Path(tempfile.mkdtemp()) / "beethoven_op002no1-01"
package.mkdir()
manifest = bundle.to_file(package / "manifest.json", serialization="json")
sorted(path.name for path in package.iterdir())

['manifest.json', 'match_claims.parquet']

In [17]:
read = AlignmentBundle.from_file(manifest)
len(read.match_claim_table), read.get_timeline(theme.id).statements == theme.statements

(26, True)